In [5]:
%load_ext autoreload
%autoreload 2
from experiment import find_all_datasets, find_all_experiments

model = "resnet"
ds_name = "cifar"
split = "trainUval"

epoch = 143 if ds_name == "cifar" else 193

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
dses = find_all_datasets("../../datasets/")
ds = dses[ds_name]

Found dataset: cifar with splits: ['train', 'trainUval', 'val']
Found dataset: emnist with splits: ['train', 'trainUval', 'val']
Adjusted labels for dataset emnist, split train to be zero-indexed.
Adjusted labels for dataset emnist, split trainUval to be zero-indexed.
Adjusted labels for dataset emnist, split val to be zero-indexed.
Found dataset: emnist_balanced with splits: ['train', 'trainUval', 'val']
Adjusted labels for dataset emnist_balanced, split train to be zero-indexed.
Adjusted labels for dataset emnist_balanced, split trainUval to be zero-indexed.
Adjusted labels for dataset emnist_balanced, split val to be zero-indexed.
Found dataset: emnist_letters with splits: ['train', 'trainUval', 'val']
Adjusted labels for dataset emnist_letters, split train to be zero-indexed.
Adjusted labels for dataset emnist_letters, split trainUval to be zero-indexed.
Adjusted labels for dataset emnist_letters, split val to be zero-indexed.
Found dataset: imagenet with splits: ['train', 'trainUv

In [7]:
data_dir = "C:/home/eurovis_data/landscape_data_cross_epoch_better/"
strees_dir = "C:/home/eurovis_data/strees_cross_epoch_better/"

In [15]:
exps = find_all_experiments(dses, data_dir, strees_dir)
exp = exps[0]
for e in exps:
	if e.model == model and e.dataset.name == ds_name and e.split == split and e.epoch == epoch:
		exp = e
		break

assert exp.model == model
assert exp.dataset.name == ds_name
assert exp.split == split
assert exp.epoch == epoch

e

resnet (cifar-trainUval), k=20, layer=1, epoch=143

In [16]:
from basic_utils import get_labels, get_partition, get_tree, get_order_and_weights
labels = get_labels(exp)
partition = get_partition(exp)

assert len(partition) == len(labels)

In [36]:
import pyct as ct

data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

homo_prop = 0.99
fns, counts = simpl.getHomoValleyPlot(order, wts, labels, homo_prop, partition) # type: ignore
fns_norm, counts_norm_min, counts_norm_max = simpl.getSimplificationPlot(order, wts)

In [37]:
import plotly.express as px

px.line(x=fns, y=counts, labels={"x": "Function Value", "y": "Number of Homogeneous Valleys"}, title=f"Homogeneous Valleys vs Function Value (Homogeneity Threshold = {homo_prop})")

In [38]:
px.line(x=fns_norm, y=counts_norm_min, labels={"x": "Function Value", "y": "Number of Valleys"}, title="Number of Valleys vs Function Value")

In [39]:
data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

fns_more, remaining_all, remaining_homo, maj_class_homo_cov, maj_class_homo_counts, class_homo_covs, class_all_covs = simpl.getHomoValleyPlotPlusCoverages(order, wts, labels, homo_prop, partition)

In [40]:
list(zip(maj_class_homo_cov, class_all_covs))

[([0.010166666666666666,
   0.011166666666666667,
   0.010666666666666666,
   0.007666666666666666,
   0.007666666666666666,
   0.007666666666666666,
   0.011,
   0.008166666666666666,
   0.011666666666666667,
   0.008666666666666666],
  [0.010166666666666666,
   0.011166666666666667,
   0.010666666666666666,
   0.007666666666666666,
   0.007666666666666666,
   0.007666666666666666,
   0.011,
   0.008166666666666666,
   0.011666666666666667,
   0.008666666666666666]),
 ([0.010166666666666666,
   0.011166666666666667,
   0.010666666666666666,
   0.007666666666666666,
   0.007666666666666666,
   0.007666666666666666,
   0.011,
   0.008166666666666666,
   0.011666666666666667,
   0.0085],
  [0.010166666666666666,
   0.011166666666666667,
   0.010666666666666666,
   0.007666666666666666,
   0.007666666666666666,
   0.007666666666666666,
   0.011,
   0.008166666666666666,
   0.011666666666666667,
   0.0085]),
 ([0.010166666666666666,
   0.011166666666666667,
   0.010666666666666666,
   0.00

In [41]:
idx = remaining_all.index(10)
idx, fns[idx], maj_class_homo_cov[idx], class_all_covs[idx], maj_class_homo_counts[idx]

(286,
 1.8834656657418236e-05,
 [0.914,
  0.9476666666666667,
  0.6585,
  0.8451666666666666,
  0.8475,
  0.8298333333333333,
  0.8825,
  0.9138333333333334,
  0.9355,
  0.7048333333333333],
 [0.9141666666666667,
  0.9476666666666667,
  0.6585,
  0.8451666666666666,
  0.8475,
  0.8298333333333333,
  0.8826666666666667,
  0.9138333333333334,
  0.9355,
  0.7048333333333333],
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [59]:
classes_needed = 10 # 10 classes, the following criteria must apply to this many of them
maj_coverage_needed = 0.5 # at least 50% coverage in homogeneous valleys where they are the only class (100% proportion in valley) 
total_coverage_needed = 0.6 # at least 60% total coverage in all valleys

# coverages are functions of accuracy, so this constraint necessarily tightens in more complex datasets

represented = [[mc >= maj_coverage_needed and tc >= total_coverage_needed for mc, tc in zip(maj_cov, total_cov)].count(True) >= classes_needed 
            	for maj_cov, total_cov in zip(maj_class_homo_cov, class_all_covs)]

# find interval where this is true
last_idx = len(fns) - represented[::-1].index(True) - 1

first_idx = represented.index(True)

print(f"Uniform Interval: {all(represented[first_idx:last_idx+1])}")
print(f"First IDX: {first_idx} - ")
print(first_idx, len(fns), fns[first_idx], list(zip(maj_class_homo_cov[first_idx], class_all_covs[first_idx], maj_class_homo_counts[first_idx])), sep="\n")
print(f"\nLast IDX: {last_idx} - ")
print(last_idx, len(fns), fns[last_idx], list(zip(maj_class_homo_cov[last_idx], class_all_covs[last_idx], maj_class_homo_counts[last_idx])), sep="\n")

Uniform Interval: True
First IDX: 286 - 
286
296
1.8834656657418236e-05
[(0.914, 0.9141666666666667, 1), (0.9476666666666667, 0.9476666666666667, 1), (0.6585, 0.6585, 1), (0.8451666666666666, 0.8451666666666666, 1), (0.8475, 0.8475, 1), (0.8298333333333333, 0.8298333333333333, 1), (0.8825, 0.8826666666666667, 1), (0.9138333333333334, 0.9138333333333334, 1), (0.9355, 0.9355, 1), (0.7048333333333333, 0.7048333333333333, 1)]

Last IDX: 286 - 
286
296
1.8834656657418236e-05
[(0.914, 0.9141666666666667, 1), (0.9476666666666667, 0.9476666666666667, 1), (0.6585, 0.6585, 1), (0.8451666666666666, 0.8451666666666666, 1), (0.8475, 0.8475, 1), (0.8298333333333333, 0.8298333333333333, 1), (0.8825, 0.8826666666666667, 1), (0.9138333333333334, 0.9138333333333334, 1), (0.9355, 0.9355, 1), (0.7048333333333333, 0.7048333333333333, 1)]
